In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd 
import time


In [2]:
def get_full_article(article_url):
        response = requests.get(article_url, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        article_content = soup.find("div", class_="entry-content")

        if article_content:
            paragraphs = article_content.find_all("p")
            full_text = " ".join(
                p.get_text(strip=True) for p in paragraphs
            )
            return full_text

In [3]:
def single_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text , 'html.parser')
    
    titles = soup.find_all('h2', class_ = "entry-title post-title")
    authors = soup.find_all('span', class_="entry-author")
    dates = soup.find_all('span', class_="entry-date")
    news_detail = soup.find_all('div', class_ = "entry-content")

    page_news = []
    for title, author, date, detail in zip(titles, authors, dates, news_detail):
        article_url = title.find("a")["href"]
        p_tag = detail.select_one('div > p:not([class])')
        full_detail = get_full_article(article_url)
        page_news.append({
            'Title': title.get_text(strip=True),
            'Author': author.get_text(strip=True),
            'Date': date.get_text(strip=True),
            'URL': article_url,
            'Detail': full_detail,
        })
        time.sleep(1)
    return page_news
            

In [4]:
def multiple_page(total_page = 3): 
    all_news = []
    for page in range(1, total_page+1):
        if page == 1:
            url = "https://mereja.net/amharic/"
        else:
            url = f"https://mereja.net/amharic/page/{page}/"
        all_news.extend(single_page(url))
        time.sleep(1)
    return pd.DataFrame(all_news)


In [5]:
df = multiple_page()
# df

In [8]:
df.to_csv("mereja.csv", index= False, encoding = "utf-8-sig")

In [ ]:
print(df.head())